In [5]:
import pickle
with open("id_to_symbol_map.pickle", "rb") as f:
    obj = pickle.load(f)

#print(obj)

In [7]:
import pandas as pd
import numpy as np 

# CPX-CPX
cpx_cpx = pd.read_csv("matt_table.csv")
cpx_cpx = cpx_cpx.rename(columns={'Node 1 Name': 'node1','Node 2 Name': 'node2'})
cpx_cpx["node1"] = np.where(cpx_cpx["node1"].eq("UNKNOWN"), "UNKNOWN_" + cpx_cpx["Node 1 ID"].astype(str), cpx_cpx["node1"])
cpx_cpx["node2"] = np.where(cpx_cpx["node2"].eq("UNKNOWN"), "UNKNOWN_" + cpx_cpx["Node 2 ID"].astype(str), cpx_cpx["node2"])
cpx_cpx = cpx_cpx[(cpx_cpx["Node 1 Tissue"] == "Choroid Plexus") &(cpx_cpx["Node 2 Tissue"] == "Choroid Plexus")]
cpx_cpx = cpx_cpx[['node1', 'node2']]
cpx_cpx["node_min"] = cpx_cpx[["node1", "node2"]].min(axis=1)
cpx_cpx["node_max"] = cpx_cpx[["node1", "node2"]].max(axis=1)
cpx_cpx = cpx_cpx.drop_duplicates(subset=["node_min", "node_max"])
cpx_cpx = cpx_cpx.drop(columns=["node_min", "node_max"])
cpx_cpx = cpx_cpx.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
cpx_cpx[['node1', 'node2']] = cpx_cpx[['node1', 'node2']].astype(str) + '-C'
cpx_cpx.to_csv("CPX-CPX_updated.csv", index=False)
print("CPX-CPX",len(cpx_cpx))

# CPX-CSF
cpx_csf = pd.read_csv("../Correlations/CSF-CPX/9. Effect Sign Prediction/new_csf-cpx_node_pairs.csv")
cpx_csf = cpx_csf.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
cpx_csf['node1'] = cpx_csf['node1'].map(obj)
cpx_csf['node1'] = cpx_csf['node1'].astype(str) + '-C'
cpx_csf['node2'] = cpx_csf['node2'].astype(str) + '-F'
cpx_csf.to_csv("CPX-CSF_updated.csv", index=False)
print("CPX-CSF",len(cpx_csf))

# CSF-CSF
csf_csf = pd.read_csv("../Correlations/CSF-CSF/5. After ESP/new_csf-csf_node_pairs.csv")
csf_csf = csf_csf.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
csf_csf = csf_csf[['node1', 'node2']]
csf_csf[['node1', 'node2']] = csf_csf[['node1', 'node2']].astype(str) + '-F'
csf_csf.to_csv("CSF-CSF_updated.csv", index=False)
print("CSF-CSF",len(csf_csf))

# CTX-CTX
ctx_ctx = pd.read_csv("matt_table.csv")
ctx_ctx = ctx_ctx.rename(columns={'Node 1 Name': 'node1','Node 2 Name': 'node2'})
ctx_ctx["node1"] = np.where(ctx_ctx["node1"].eq("UNKNOWN"), "UNKNOWN_" + ctx_ctx["Node 1 ID"].astype(str), ctx_ctx["node1"])
ctx_ctx["node2"] = np.where(ctx_ctx["node2"].eq("UNKNOWN"), "UNKNOWN_" + ctx_ctx["Node 2 ID"].astype(str), ctx_ctx["node2"])
ctx_ctx = ctx_ctx[(ctx_ctx["Node 1 Tissue"] == "Cortex") &(ctx_ctx["Node 2 Tissue"] == "Cortex")]
ctx_ctx = ctx_ctx[['node1', 'node2']]
ctx_ctx["node_min"] = ctx_ctx[["node1", "node2"]].min(axis=1)
ctx_ctx["node_max"] = ctx_ctx[["node1", "node2"]].max(axis=1)
ctx_ctx = ctx_ctx.drop_duplicates(subset=["node_min", "node_max"])
ctx_ctx = ctx_ctx.drop(columns=["node_min", "node_max"])
ctx_ctx = ctx_ctx.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
ctx_ctx[['node1', 'node2']] = ctx_ctx[['node1', 'node2']].astype(str) + '-T'
ctx_ctx.to_csv("CTX-CTX_updated.csv", index=False)
print("CTX-CTX",len(ctx_ctx))

# CSF-CTX
csf_ctx = pd.read_csv("../Correlations/final_network.csv")
csf_ctx = csf_ctx[csf_ctx["predicted = pooled"].notna()]
csf_ctx = csf_ctx[csf_ctx["type_brain"] == "CTX"]
csf_ctx = csf_ctx[csf_ctx["Pooled FDR"] < 0.15]
csf_ctx = csf_ctx.rename(columns={'brain_name': 'node1','CSF_name': 'node2'})
csf_ctx['node1'] = csf_ctx['node1'].str.replace(r'-C$', '', regex=True)
csf_ctx['node1'] = csf_ctx['node1'].astype(str).str.strip()
csf_ctx['node2'] = csf_ctx['node2'].astype(str).str.strip()
csf_ctx = csf_ctx[['node1', 'node2']]
csf_ctx = csf_ctx[['node1', 'node2']]
# manual overrides for raw IDs that `obj` maps to a bare "UNKNOWN"
unknown_overrides = {
    "ENSMUSG00000053545": "UNKNOWN_1343",
    "ENSMUSG00000078091": "UNKNOWN_1454",
}
csf_ctx['node1'] = csf_ctx['node1'].replace(unknown_overrides).map(
    lambda x: x if x in unknown_overrides.values() else obj.get(x, x))
csf_ctx['node1'] = csf_ctx['node1'].astype(str) + '-T'
csf_ctx['node2'] = csf_ctx['node2'].astype(str) + '-F'
csf_ctx.to_csv("CSF-CTX_updated.csv", index=False)
print("CSF-CTX",len(csf_ctx))
# ENSMUSG00000053545 = UNKNOWN_1343
# ENSMUSG00000078091 = UNKNOWN_1454



# STM-STM
stm_stm = pd.read_csv("matt_table.csv")
stm_stm = stm_stm.rename(columns={'Node 1 Name': 'node1','Node 2 Name': 'node2'})
stm_stm["node1"] = np.where(stm_stm["node1"].eq("UNKNOWN"), "UNKNOWN_" + stm_stm["Node 1 ID"].astype(str), stm_stm["node1"])
stm_stm["node2"] = np.where(stm_stm["node2"].eq("UNKNOWN"), "UNKNOWN_" + stm_stm["Node 2 ID"].astype(str), stm_stm["node2"])
stm_stm = stm_stm[(stm_stm["Node 1 Tissue"] == "Striatum") &(stm_stm["Node 2 Tissue"] == "Striatum")]
stm_stm = stm_stm[['node1', 'node2']]
stm_stm["node_min"] = stm_stm[["node1", "node2"]].min(axis=1)
stm_stm["node_max"] = stm_stm[["node1", "node2"]].max(axis=1)
stm_stm = stm_stm.drop_duplicates(subset=["node_min", "node_max"])
stm_stm = stm_stm.drop(columns=["node_min", "node_max"])
stm_stm = stm_stm.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
stm_stm[['node1', 'node2']] = stm_stm[['node1', 'node2']].astype(str) + '-S'
stm_stm.to_csv("STM-STM_updated.csv", index=False)
print("STM-STM",len(stm_stm))


# CSF-STM
csf_stm = pd.read_csv("../Correlations/final_network.csv")
csf_stm = csf_stm[csf_stm["predicted = pooled"].notna()]
csf_stm = csf_stm[csf_stm["type_brain"] == "STM"]
csf_stm = csf_stm[csf_stm["Pooled FDR"] < 0.15]
csf_stm = csf_stm.rename(columns={'brain_name': 'node1','CSF_name': 'node2'})
csf_stm['node1'] = csf_stm['node1'].str.replace(r'-S$', '', regex=True)
csf_stm['node1'] = csf_stm['node1'].astype(str).str.strip()
csf_stm['node2'] = csf_stm['node2'].astype(str).str.strip()
csf_stm = csf_stm[['node1', 'node2']]
csf_stm['node1'] = csf_stm['node1'].map(obj)
csf_stm['node1'] = csf_stm['node1'].astype(str) + '-S'
csf_stm['node2'] = csf_stm['node2'].astype(str) + '-F'
csf_stm.to_csv("CSF-STM_updated.csv", index=False)
print("CSF-STM",len(csf_stm))

# RBN-RBN
rbn_rbn = pd.read_csv("matt_table.csv")
rbn_rbn = rbn_rbn.rename(columns={'Node 1 Name': 'node1','Node 2 Name': 'node2'})
rbn_rbn["node1"] = np.where(rbn_rbn["node1"].eq("UNKNOWN"), "UNKNOWN_" + rbn_rbn["Node 1 ID"].astype(str), rbn_rbn["node1"])
rbn_rbn["node2"] = np.where(rbn_rbn["node2"].eq("UNKNOWN"), "UNKNOWN_" + rbn_rbn["Node 2 ID"].astype(str), rbn_rbn["node2"])
rbn_rbn = rbn_rbn[(rbn_rbn["Node 1 Tissue"] == "Remaining Brain") &(rbn_rbn["Node 2 Tissue"] == "Remaining Brain")]
rbn_rbn = rbn_rbn[['node1', 'node2']]
rbn_rbn["node_min"] = rbn_rbn[["node1", "node2"]].min(axis=1)
rbn_rbn["node_max"] = rbn_rbn[["node1", "node2"]].max(axis=1)
rbn_rbn = rbn_rbn.drop_duplicates(subset=["node_min", "node_max"])
rbn_rbn = rbn_rbn.drop(columns=["node_min", "node_max"])
rbn_rbn = rbn_rbn.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
rbn_rbn[['node1', 'node2']] = rbn_rbn[['node1', 'node2']].astype(str) + '-B'
rbn_rbn.to_csv("RBN-RBN_updated.csv", index=False)
print("RBN-RBN",len(rbn_rbn))

# CSF-RBN
csf_rbn = pd.read_csv("../Correlations/final_network.csv")
csf_rbn = csf_rbn[csf_rbn["predicted = pooled"].notna()]
csf_rbn = csf_rbn[csf_rbn["type_brain"] == "RBN"]
csf_rbn = csf_rbn[csf_rbn["Pooled FDR"] < 0.15]
csf_rbn = csf_rbn.rename(columns={'brain_name': 'node1','CSF_name': 'node2'})
csf_rbn['node1'] = csf_rbn['node1'].str.replace(r'-B$', '', regex=True)
csf_rbn['node1'] = csf_rbn['node1'].astype(str).str.strip()
csf_rbn['node2'] = csf_rbn['node2'].astype(str).str.strip()
csf_rbn = csf_rbn[['node1', 'node2']]
csf_rbn['node1'] = csf_rbn['node1'].map(obj)
csf_rbn['node1'] = csf_rbn['node1'].astype(str) + '-B'
csf_rbn['node2'] = csf_rbn['node2'].astype(str) + '-F'
csf_rbn.to_csv("CSF-RBN_updated.csv", index=False)
print("CSF-RBN",len(csf_rbn))





/var/folders/0q/ndw_qsrd5cbch1pd347wrwx40000gn/T/ipykernel_27366/3977241823.py:5: DtypeWarning: Columns (43,45,56,102) have mixed types. Specify dtype option on import or set low_memory=False.
  cpx_cpx = pd.read_csv("matt_table.csv")


CPX-CPX 4069
CPX-CSF 163
CSF-CSF 313


/var/folders/0q/ndw_qsrd5cbch1pd347wrwx40000gn/T/ipykernel_27366/3977241823.py:38: DtypeWarning: Columns (43,45,56,102) have mixed types. Specify dtype option on import or set low_memory=False.
  ctx_ctx = pd.read_csv("matt_table.csv")


CTX-CTX 2570
CSF-CTX 494


/var/folders/0q/ndw_qsrd5cbch1pd347wrwx40000gn/T/ipykernel_27366/3977241823.py:81: DtypeWarning: Columns (43,45,56,102) have mixed types. Specify dtype option on import or set low_memory=False.
  stm_stm = pd.read_csv("matt_table.csv")


STM-STM 1890
CSF-STM 518


/var/folders/0q/ndw_qsrd5cbch1pd347wrwx40000gn/T/ipykernel_27366/3977241823.py:114: DtypeWarning: Columns (43,45,56,102) have mixed types. Specify dtype option on import or set low_memory=False.
  rbn_rbn = pd.read_csv("matt_table.csv")


RBN-RBN 194
CSF-RBN 248


In [8]:
# after cpx_cpx is built (before it's overwritten with the -C suffix, or after — either works)
def show_unknowns(df, label):
    unk1 = df.loc[df["node1"].astype(str).str.startswith("UNKNOWN"), "node1"]
    unk2 = df.loc[df["node2"].astype(str).str.startswith("UNKNOWN"), "node2"]
    all_unk = pd.concat([unk1, unk2]).unique()
    print(f"--- {label}: {len(all_unk)} unique UNKNOWN ids ---")
    for u in sorted(all_unk):
        print(u)
    print()

show_unknowns(cpx_cpx, "CPX-CPX")
show_unknowns(csf_ctx, "CSF-CTX")

show_unknowns(ctx_ctx, "CTX-CTX")
show_unknowns(stm_stm, "STM-STM")
show_unknowns(rbn_rbn, "RBN-RBN")

--- CPX-CPX: 1 unique UNKNOWN ids ---
UNKNOWN_158-C

--- CSF-CTX: 2 unique UNKNOWN ids ---
UNKNOWN-T
UNKNOWN_1454-T

--- CTX-CTX: 2 unique UNKNOWN ids ---
UNKNOWN_1345-T
UNKNOWN_1454-T

--- STM-STM: 0 unique UNKNOWN ids ---

--- RBN-RBN: 1 unique UNKNOWN ids ---
UNKNOWN_2821-B



In [11]:
import pandas as pd

files = [
    "CPX-CPX_updated.csv",
    "CPX-CSF_updated.csv",
    "CSF-CSF_updated.csv",
    
    "CTX-CTX_updated.csv",
    "CSF-CTX_updated.csv",
    
    "STM-STM_updated.csv",
    "CSF-STM_updated.csv",
    
    "RBN-RBN_updated.csv",
    "CSF-RBN_updated.csv",
]

combined = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

# strip whitespace from ALL string cells
combined = combined.apply(
    lambda col: col.str.strip() if col.dtype == "object" else col
)

print("Total edges", len(combined))
unique_nodes = pd.unique(combined[["node1", "node2"]].values.ravel())
print("Number of unique nodes:", len(unique_nodes))

combined.to_csv("cpx_brain_final_edges_table.csv", index=False)

Total edges 10459
Number of unique nodes: 2385


In [13]:
import pandas as pd

node_type = pd.DataFrame(columns=["Node", "Type"])


# CPX-CPX
df_cc = pd.read_csv("CPX-CPX_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_cc["node1"], "Type": "CPX"}),
    pd.DataFrame({"Node": df_cc["node2"], "Type": "CPX"})
], ignore_index=True)

# CPX-CSF
df_cf = pd.read_csv("CPX-CSF_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_cf["node1"], "Type": "CPX"}),
    pd.DataFrame({"Node": df_cf["node2"], "Type": "CSF"})
], ignore_index=True)

# CSF-CSF
df_ff = pd.read_csv("CSF-CSF_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_ff["node1"], "Type": "CSF"}),
    pd.DataFrame({"Node": df_ff["node2"], "Type": "CSF"})
], ignore_index=True)

# CTX-CTX
df_tt = pd.read_csv("CTX-CTX_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_tt["node1"], "Type": "CTX"}),
    pd.DataFrame({"Node": df_tt["node2"], "Type": "CTX"})
], ignore_index=True)

# CSF-CTX
df_ft = pd.read_csv("CSF-CTX_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_ft["node1"], "Type": "CTX"}),
    pd.DataFrame({"Node": df_ft["node2"], "Type": "CSF"})
], ignore_index=True)

# STM-STM
df_ss = pd.read_csv("STM-STM_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_ss["node1"], "Type": "STM"}),
    pd.DataFrame({"Node": df_ss["node2"], "Type": "STM"})
], ignore_index=True)

# CSF-STM
df_fs = pd.read_csv("CSF-STM_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_fs["node1"], "Type": "STM"}),
    pd.DataFrame({"Node": df_fs["node2"], "Type": "CSF"})
], ignore_index=True)

# RBN-RBN
df_rr = pd.read_csv("RBN-RBN_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_rr["node1"], "Type": "RBN"}),
    pd.DataFrame({"Node": df_rr["node2"], "Type": "RBN"})
], ignore_index=True)


# CSF-RBN
df_fr = pd.read_csv("CSF-RBN_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_fr["node1"], "Type": "RBN"}),
    pd.DataFrame({"Node": df_fr["node2"], "Type": "CSF"})
], ignore_index=True)


node_type = node_type.map(lambda x: x.strip() if isinstance(x, str) else x)
node_type = node_type.drop_duplicates().sort_values("Node").reset_index(drop=True)
print(len(node_type))

node_type.to_csv("cpx_brain_node_types.csv", index=False, header=False)



# ----------------------------------------------------------------------------------


2385


In [15]:
import pandas as pd

node_type = pd.DataFrame(columns=["Node", "Type"])


# CPX-CPX
df_cc = pd.read_csv("CPX-CPX_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_cc["node1"], "Type": "CPX"}),
    pd.DataFrame({"Node": df_cc["node2"], "Type": "CPX"})
], ignore_index=True)

# CPX-CSF
df_cf = pd.read_csv("CPX-CSF_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_cf["node1"], "Type": "CPX"}),
    pd.DataFrame({"Node": df_cf["node2"], "Type": "CSF"})
], ignore_index=True)

# CSF-CSF
df_ff = pd.read_csv("CSF-CSF_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_ff["node1"], "Type": "CSF"}),
    pd.DataFrame({"Node": df_ff["node2"], "Type": "CSF"})
], ignore_index=True)

# CTX-CTX
df_tt = pd.read_csv("CTX-CTX_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_tt["node1"], "Type": "Brain"}),
    pd.DataFrame({"Node": df_tt["node2"], "Type": "Brain"})
], ignore_index=True)

# CSF-CTX
df_ft = pd.read_csv("CSF-CTX_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_ft["node1"], "Type": "Brain"}),
    pd.DataFrame({"Node": df_ft["node2"], "Type": "CSF"})
], ignore_index=True)

# STM-STM
df_ss = pd.read_csv("STM-STM_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_ss["node1"], "Type": "Brain"}),
    pd.DataFrame({"Node": df_ss["node2"], "Type": "Brain"})
], ignore_index=True)

# CSF-STM
df_fs = pd.read_csv("CSF-STM_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_fs["node1"], "Type": "Brain"}),
    pd.DataFrame({"Node": df_fs["node2"], "Type": "CSF"})
], ignore_index=True)

# RBN-RBN
df_rr = pd.read_csv("RBN-RBN_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_rr["node1"], "Type": "Brain"}),
    pd.DataFrame({"Node": df_rr["node2"], "Type": "Brain"})
], ignore_index=True)


# CSF-RBN
df_fr = pd.read_csv("CSF-RBN_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_fr["node1"], "Type": "Brain"}),
    pd.DataFrame({"Node": df_fr["node2"], "Type": "CSF"})
], ignore_index=True)


node_type = node_type.map(lambda x: x.strip() if isinstance(x, str) else x)
node_type = node_type.drop_duplicates().sort_values("Node").reset_index(drop=True)
print(len(node_type))

node_type.to_csv("cpx_{STM,CTX,RBN}_node_types.csv", index=False, header=False)



# ----------------------------------------------------------------------------------


2385
